In [1]:
"""measure_kan_runtime.py – *absolute-RSS KAN models*

Benchmarks inference runtime for **Kernel Adaptive Network (KAN)** checkpoints
trained on 5‑dim RSS inputs and predicting (X, Y, Z).

The stopwatch encloses:

1. Forward batch pass `model(test_tensor)` (CPU)
2. Descaling      inverse‑transform with `scaler_absolute.pkl`

Predictions are discarded; an optional CSV with timing metadata can be written.

Configuration
-------------
* **Model folders** `models/KANs_absolute/KAN_model_rss_n=400_noise=0.0_seed={seed}`
  *Checkpoint file* `0.3_state` (assumed final grid = 10, width = [5, 3, 3], k = 3)
* **Test set** `3D-Data/Measured_Normalized/gnd_n=50_noise=0.0.csv`
  → 125 000 rows (50 × 2500) per benchmark.
* **Scaler** `scaler_absolute.pkl`
* **Seeds scanned** 0 … 9 (only folders containing `0.3_state` are benchmarked).

Notes
-----
* Runs **CPU‑only** (KAN isn’t GPU‑accelerated yet).
* No predictions or models are saved/modified.
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from kan import KAN  # pip install kan (or your local clone)
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
MODEL_ROOT = Path("models/KANs_absolute")
SCALER_PATH = Path("scaler_absolute.pkl")

CATEGORY = "Measured_Normalized"
BASE_DIR = Path("3D-Data") / CATEGORY

TRAIN_N = 400     # only benchmark models trained on this sample size
NOISE = 0.0       # training + test noise
TEST_GND_N = 50   # rows per gnd CSV

SEEDS = range(10)

SAVE_CSV = True
CSV_PATH = Path("kan_runtime_results.csv")

DEVICE = torch.device("cpu")

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def build_ckpt_path(seed: int) -> Path:
    folder = MODEL_ROOT / f"KAN_model_rss_n={TRAIN_N}_noise={NOISE}_seed={seed}"
    ckpt = folder / "0.3_state"  # final refined checkpoint
    return ckpt if ckpt.exists() else None


def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / f"gnd_n={TEST_GND_N}_noise={NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


INPUT_COLS = [f"RSS{i}" for i in range(5)]


def build_model() -> KAN:
    """Construct a fresh KAN with the architecture used in training."""
    width = [len(INPUT_COLS), 3, 3]  # 5 → 3 → 3 (XYZ)
    model = KAN(width=width, grid=10, k=3, seed=1, device=DEVICE, auto_save=False)
    return model


def forward_and_descale(model: KAN, rss_np: np.ndarray, scaler: StandardScaler | None):
    with torch.no_grad():
        preds = model(torch.tensor(rss_np, dtype=torch.float32, device=DEVICE)).cpu().numpy()

    if scaler is None:
        return preds

    zeros = np.zeros_like(rss_np)
    descaled = scaler.inverse_transform(np.hstack([zeros, preds]))[:, -3:]
    return descaled

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)
RSS_TEST = TEST_DF[INPUT_COLS].values.astype(np.float32)

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking KAN models (n={TRAIN_N}, noise=0.0) on {N_TEST} samples\n"
)

timings = []

for seed in SEEDS:
    ckpt = build_ckpt_path(seed)
    if ckpt is None:
        continue

    model = build_model()
    state = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()

    start = time.perf_counter()
    _ = forward_and_descale(model, RSS_TEST, SCALER)
    dt = time.perf_counter() - start

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.2f} s | per-sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No matching KAN checkpoints found!")
else:
    df = pd.DataFrame(timings)
    if SAVE_CSV:
        df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …
Benchmarking KAN models (n=400, noise=0.0) on 125000 samples



/tmp/ipykernel_2201634/1987806338.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  0 | 125000 predictions | total: 0.48 s | per-sample: 0.000004 s


/tmp/ipykernel_2201634/1987806338.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  1 | 125000 predictions | total: 0.32 s | per-sample: 0.000003 s


...

Seed  7 | 125000 predictions | total: 0.31 s | per-sample: 0.000003 s


/tmp/ipykernel_2201634/1987806338.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  8 | 125000 predictions | total: 0.31 s | per-sample: 0.000002 s


/tmp/ipykernel_2201634/1987806338.py:124: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  9 | 125000 predictions | total: 0.32 s | per-sample: 0.000003 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/kan_runtime_results.csv

Done.


In [2]:
"""measure_kan_runtime.py – *relative-RSS KAN models*

Benchmarks inference runtime for **Kernel Adaptive Network (KAN)** checkpoints
trained on the 10‑dim *relative* RSS feature set (e.g. RSS0/RSS1, …) and
predicting (X, Y, Z).

The stopwatch encloses:

1. Forward batch pass `model(test_tensor)` (CPU)
2. Descaling      inverse‑transform with `scaler_relative.pkl`

Predictions are discarded; an optional CSV with timing metadata can be written.

Configuration
-------------
* **Model folders** `models/KANs_relative/KAN_model_relative_rss_n=400_noise=0.0_seed={seed}`
  *Checkpoint file* `0.3_state` (final grid = 10, width = [10, 3, 3], k = 3)
* **Test set** `3D-Data/Measured_relative_Normalized/relative_gnd_n=50_noise=0.0.csv`
  → 125 000 rows per benchmark.
* **Scaler** `scaler_relative.pkl`
* **Seeds scanned** 0 … 9 (only folders containing `0.3_state` are benchmarked).

Notes
-----
* Runs **CPU‑only**.
* No predictions or models are saved/modified.
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from kan import KAN
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
MODEL_ROOT = Path("models/KANs_relative")
SCALER_PATH = Path("scaler_relative.pkl")

CATEGORY = "Measured_relative_Normalized"
BASE_DIR = Path("3D-Data") / CATEGORY

TRAIN_N = 400     # benchmark models trained on this sample size
NOISE = 0.0       # training + test noise
TEST_GND_N = 50   # rows per gnd CSV

SEEDS = range(10)

SAVE_CSV = True
CSV_PATH = Path("kan_runtime_results_relative.csv")

DEVICE = torch.device("cpu")

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def build_ckpt_path(seed: int) -> Path | None:
    folder_name = f"KAN_model_relative_rss_n={TRAIN_N}_noise={NOISE}_seed={seed}"
    folder = MODEL_ROOT / folder_name
    ckpt = folder / "0.3_state"
    return ckpt if ckpt.exists() else None


def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / f"relative_gnd_n={TEST_GND_N}_noise={NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)


def build_model(input_dim: int) -> KAN:
    """Construct a fresh KAN with appropriate input dimension."""
    width = [input_dim, 3, 3]  # input_dim → 3 → XYZ
    return KAN(width=width, grid=10, k=3, seed=1, device=DEVICE, auto_save=False)


def forward_and_descale(model: KAN, rss_np: np.ndarray, scaler: StandardScaler | None):
    with torch.no_grad():
        preds = model(torch.tensor(rss_np, dtype=torch.float32, device=DEVICE)).cpu().numpy()

    if scaler is None:
        return preds

    zeros = np.zeros_like(rss_np)
    descaled = scaler.inverse_transform(np.hstack([zeros, preds]))[:, -3:]
    return descaled

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

TEST_DF = load_test_df()
OUTPUT_COLS = ["X", "Y", "Z"]
INPUT_COLS = [c for c in TEST_DF.columns if c not in OUTPUT_COLS]
N_TEST = len(TEST_DF)
RSS_TEST = TEST_DF[INPUT_COLS].values.astype(np.float32)
INPUT_DIM = len(INPUT_COLS)

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking relative‑RSS KAN models (n={TRAIN_N}, noise=0.0) on {N_TEST} samples\n"
)

timings = []

for seed in SEEDS:
    ckpt = build_ckpt_path(seed)
    if ckpt is None:
        continue

    model = build_model(INPUT_DIM)
    state = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()

    start = time.perf_counter()
    _ = forward_and_descale(model, RSS_TEST, SCALER)
    dt = time.perf_counter() - start

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.2f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No matching relative KAN checkpoints found!")
else:
    df = pd.DataFrame(timings)
    if SAVE_CSV:
        df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …
Benchmarking relative‑RSS KAN models (n=400, noise=0.0) on 125000 samples



/tmp/ipykernel_2201634/822425527.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  0 | 125000 predictions | total: 0.53 s | per‑sample: 0.000004 s


/tmp/ipykernel_2201634/822425527.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  1 | 125000 predictions | total: 0.53 s | per‑sample: 0.000004 s


...

Seed  7 | 125000 predictions | total: 0.58 s | per‑sample: 0.000005 s


/tmp/ipykernel_2201634/822425527.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  8 | 125000 predictions | total: 0.53 s | per‑sample: 0.000004 s


/tmp/ipykernel_2201634/822425527.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  9 | 125000 predictions | total: 0.53 s | per‑sample: 0.000004 s

Timing table written to → /home/sumo/anikraft/RSS/chapter4_recreating_results/kan_runtime_results_relative.csv

Done.


In [3]:
"""measure_kan_runtime.py – *log‑distance KAN models*

Benchmarks end‑to‑end inference runtime for **Kernel Adaptive Network (KAN)**
checkpoints that:

1. Predict five **log‑distances** (D0 … D4) from 5‑dim RSS inputs.
2. Inverse‑scale the log‑distances with `scaler_log_d.pkl`.
3. Exponentiate → linear metres.
4. Trilaterate (non‑linear least‑squares) to recover (X, Y, Z).

The stopwatch encloses **all four stages**. Predictions are discarded; only an
optional timing CSV is produced.

Configuration
-------------
* **Model folders** `models/KANs_log_d/KAN_model_rss_n=400_noise=0.0_seed={seed}`
  *Checkpoint file* `0.3_state` (final grid = 10, width = [5, 3, 5], k = 3)
* **Test set** `3D-Data/Measured_log_d_Normalized/gnd_n=50_noise=0.0.csv`
  → 125 000 rows per benchmark.
* **Scaler** `scaler_log_d.pkl`
* **Seeds scanned** 0 … 9 (only folders containing `0.3_state` are benchmarked).

Notes
-----
* Runs **CPU‑only**.
* No predictions or models are saved/modified.
"""

from __future__ import annotations

import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from kan import KAN
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------------------------------------------
# Configuration ------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
MODEL_ROOT = Path("models/KANs_log_d")
SCALER_PATH = Path("scaler_log_d.pkl")

CATEGORY = "Measured_log_d_Normalized"
BASE_DIR = Path("3D-Data") / CATEGORY

TRAIN_N = 400     # benchmark models trained on this sample size
NOISE = 0.0       # training + test noise
TEST_GND_N = 50   # rows per gnd CSV

SEEDS = range(10)

SAVE_CSV = True
CSV_PATH = Path("kan_runtime_results_logd.csv")

DEVICE = torch.device("cpu")

# --------------------------------------------------------------------------------------
# Trilateration helpers ----------------------------------------------------------------
# --------------------------------------------------------------------------------------
BEACON_COORDS = np.array(
    [
        [0, 0, 5],
        [-2, -2, 5],
        [-2,  2, 5],
        [2, -2, 5],
        [2,  2, 5],
    ]
)

def _mse_error(pos, beacon_coords, distances):
    pred = np.linalg.norm(beacon_coords - pos, axis=1)
    return np.mean((pred - distances) ** 2)


def trilaterate_mse(distances, beacon_coords=BEACON_COORDS):
    guess = np.mean(beacon_coords, axis=0)
    bounds = [(None, None), (None, None), (None, 5)]
    res = minimize(_mse_error, guess, args=(beacon_coords, distances), method="L-BFGS-B", bounds=bounds)
    if not res.success:
        raise ValueError("Trilateration failed: " + res.message)
    return res.x

# --------------------------------------------------------------------------------------
# Helper functions ---------------------------------------------------------------------
# --------------------------------------------------------------------------------------

def build_ckpt_path(seed: int) -> Path | None:
    folder = MODEL_ROOT / f"KAN_model_rss_n={TRAIN_N}_noise={NOISE}_seed={seed}"
    ckpt = folder / "0.3_state"
    return ckpt if ckpt.exists() else None


def load_test_df() -> pd.DataFrame:
    fp = BASE_DIR / f"gnd_n={TEST_GND_N}_noise={NOISE}.csv"
    if not fp.exists():
        raise FileNotFoundError(f"Test CSV not found: {fp}")
    return pd.read_csv(fp)

INPUT_COLS = [f"RSS{i}" for i in range(5)]
DIST_COLS = [f"D{i}" for i in range(5)]


def build_model() -> KAN:
    """5 RSS inputs → hidden 3 → 5 log‑distance outputs."""
    width = [len(INPUT_COLS), 3, 5]
    return KAN(width=width, grid=10, k=3, seed=1, device=DEVICE, auto_save=False)


def full_pipeline(model: KAN, rss_np: np.ndarray, scaler: StandardScaler | None):
    """Predict log‑d → inverse‑scale → exp → trilaterate."""
    with torch.no_grad():
        preds = model(torch.tensor(rss_np, dtype=torch.float32, device=DEVICE)).cpu().numpy()

    # inverse‑scale
    if scaler is not None:
        zeros = np.zeros_like(rss_np)
        preds = scaler.inverse_transform(np.hstack([zeros, preds]))[:, -5:]

    # exp → linear metres
    lin_d = np.exp(preds)

    # trilaterate row‑wise
    coords = np.empty((lin_d.shape[0], 3))
    for i, d in enumerate(lin_d):
        coords[i] = trilaterate_mse(d)

    return coords

# --------------------------------------------------------------------------------------
# Benchmark ---------------------------------------------------------------------------
# --------------------------------------------------------------------------------------
print("Loading resources …", flush=True)

TEST_DF = load_test_df()
N_TEST = len(TEST_DF)
RSS_TEST = TEST_DF[INPUT_COLS].values.astype(np.float32)

SCALER = None
if SCALER_PATH.exists():
    with SCALER_PATH.open("rb") as fh:
        SCALER = pickle.load(fh)

print(
    f"Benchmarking log‑d KAN models (n={TRAIN_N}, noise=0.0) on {N_TEST} samples\n"
)

timings = []

for seed in SEEDS:
    ckpt = build_ckpt_path(seed)
    if ckpt is None:
        continue

    model = build_model()
    state = torch.load(ckpt, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()

    start = time.perf_counter()
    _ = full_pipeline(model, RSS_TEST, SCALER)
    dt = time.perf_counter() - start

    timings.append(
        {
            "seed": seed,
            "runtime_s": dt,
            "per_sample_s": dt / N_TEST,
            "n_test": N_TEST,
        }
    )

    print(
        f"Seed {seed:2d} | {N_TEST} predictions | total: {dt:.2f} s | per‑sample: {dt/N_TEST:.6f} s"
    )

if not timings:
    print("⚠️  No matching log‑d KAN checkpoints found!")
else:
    df = pd.DataFrame(timings)
    if SAVE_CSV:
        df.to_csv(CSV_PATH, index=False)
        print(f"\nTiming table written to → {CSV_PATH.resolve()}")

print("\nDone.")


Loading resources …
Benchmarking log‑d KAN models (n=400, noise=0.0) on 125000 samples



/tmp/ipykernel_2201634/285175663.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  0 | 125000 predictions | total: 678.18 s | per‑sample: 0.005425 s


/tmp/ipykernel_2201634/285175663.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  1 | 125000 predictions | total: 678.89 s | per‑sample: 0.005431 s


...

Seed  6 | 125000 predictions | total: 676.32 s | per‑sample: 0.005411 s


/tmp/ipykernel_2201634/285175663.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


Seed  7 | 125000 predictions | total: 677.25 s | per‑sample: 0.005418 s


/tmp/ipykernel_2201634/285175663.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt, map_location=DEVICE)


KeyboardInterrupt: 